새 window CSV 예측 코드

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import files

# 모델 파일 업로드 또는 Colab에 이미 압축 해제되어 있다고 가정
model = tf.keras.models.load_model("drowsy_mlp_model/drowsy_mlp.keras")
scaler = joblib.load("drowsy_mlp_model/scaler.pkl")

with open("drowsy_mlp_model/feature_columns.json", "r", encoding="utf-8") as f:
    feature_cols = json.load(f)

uploaded = files.upload()
new_csv_path = list(uploaded.keys())[0]

new_df = pd.read_csv(new_csv_path)

# 학습 때 사용한 feature 순서 그대로 맞춤
for col in feature_cols:
    if col not in new_df.columns:
        new_df[col] = 0

X_new = new_df[feature_cols].copy()
X_new = X_new.replace([np.inf, -np.inf], np.nan).fillna(0)
X_new_scaled = scaler.transform(X_new.values.astype(np.float32))

probs = model.predict(X_new_scaled).ravel()

result_df = new_df.copy()
result_df["drowsy_probability"] = probs
result_df["prediction"] = np.where(probs >= 0.5, "drowsy", "normal")

display(result_df[["video_name", "window_id", "drowsy_probability", "prediction"]])